In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import sys
from logging import INFO, StreamHandler, getLogger

logger = getLogger()
if not logger.hasHandlers():
    logger.addHandler(StreamHandler(sys.stdout))
logger.setLevel(INFO)

# Import


In [ ]:
import os
import pathlib
import sys
from itertools import product

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.gridspec import GridSpec

ROOT_DIR = str(pathlib.Path(".").resolve().parent.parent)
print(f"ROOT_DIR: {ROOT_DIR}")
plt.rcParams["font.family"] = "serif"
plt.style.use("tableau-colorblind10")

In [ ]:
sys.path.append(ROOT_DIR)

In [ ]:
from src.util.io_pickle import read_pickle
from src.synthetic_data.data_generator import generate_correlated_uniform

# Constants


In [ ]:
RESULT_DIR = f"{ROOT_DIR}/data/synthetic_data"
FIG_DIR = f"{ROOT_DIR}/notebook/figs"
os.makedirs(FIG_DIR, exist_ok=True)

# Time series


## Uniform


In [ ]:
n_ens = 200
n_times = 1_200
low, high = -1.0, 1.0
corr = 0.25
seed = 42

config = {
    "n_ensembles": n_ens,
    "data_length": n_times,
    "low": low,
    "high": high,
    "correlation": corr,
    "seed": seed,
}

data = generate_correlated_uniform(**config)
assert data.shape == (n_ens, n_times)
means = np.mean(data, axis=0)
ts = np.linspace(0, n_times / 12.0, n_times)

In [ ]:
plt.rcParams["font.size"] = 25

fig = plt.figure(figsize=[10, 5.0])
gs = GridSpec(1, 5, figure=fig)
ax1 = fig.add_subplot(gs[:, :3])
ax2 = fig.add_subplot(gs[:, 3:])

vmin, vmax = -1.2, 1.2

interval = 24
m_bar = np.mean(means.flatten())

ax = ax1
ax.plot(
    ts[::interval],
    means[::interval],
    ls="-",
    lw=3,
    color="#006BA4",
    label="Ens. Mean",
)
for i in range(10):
    l = "All Members" if i == 0 else None
    ax.plot(
        ts[::interval],
        data[i, ::interval],
        ls="-",
        lw=1,
        alpha=0.3,
        color="#FF800E",
        label=l,
    )

ax.axhline(m_bar, ls="--", lw=2, color="k")
ax.set_xlabel("Time [yr]")
ax.set_ylabel("Climate Variable")
ax.set_ylim(vmin, vmax)

ax = ax2
ax.set_yticks([])
cnts = ax.hist(
    means.flatten(),
    density=True,
    orientation="horizontal",
    range=(vmin, vmax),
    bins=21,
    alpha=0.3,
    color="#006BA4",
    label="Ens. Mean",
    linewidth=None,
)
cnts = ax.hist(
    data.flatten(),
    density=True,
    orientation="horizontal",
    range=(vmin, vmax),
    bins=21,
    alpha=0.3,
    label="All Members",
    color="#FF800E",
    linewidth=None,
)
ax.axhline(m_bar, ls="--", lw=2, color="k")

ax.set_xlim(0, 2)
ax.set_xlabel("PDF")
ax.set_ylim(vmin, vmax)

plt.tight_layout()
fig.savefig(f"{FIG_DIR}/fig01_uniform1.jpg", bbox_inches="tight")

In [ ]:
fig, axes = plt.subplots(1, 2, sharey=True, sharex=True, figsize=[10, 2.5])
for ax, data, color in zip(axes, [means, data], ["#006BA4", "#FF800E"]):
    print(data.min(), data.max())
    ax.hist(
        data.flatten(),
        density=True,
        range=(vmin, vmax),
        bins=21,
        alpha=0.3,
        color=color,
        linewidth=None,
    )
    ax.set_xticks([])
    ax.set_yticks([])

axes[1].axvline(m_bar, ls="--", lw=2, color="k")

plt.tight_layout()
fig.savefig(f"{FIG_DIR}/fig01_uniform2.jpg", bbox_inches="tight")

# Ensenble Size vs Time-series Length


In [ ]:
dict_metric_names = {
    "gamma_std": r"$\gamma_\mathrm{var}$",
    "gamma_sane": r"$\gamma_\mathrm{info}$",
    "gamma_sane_original": r"$\gamma_\mathrm{info,org}$",
    "gamma_sane_original_with_fixed_params": r"$\gamma_\mathrm{info,org,fixed}$",
    "gamma_w1": r"$\gamma_{W_1,{\rm pot}}$",
    "gamma_w2": r"$\gamma_{W_2,{\rm pot}}$",
    "gamma_w3": r"$\gamma_{W_1}$",
    "gamma_w4": r"$\gamma_{W_2}$",
    "gamma_w5": r"$\widetilde{\gamma}_{W_1}$",
}

target_metrics = ["gamma_w3"]
cmap = "BuPu"

for is_outlier, target_metric in product([True, False], target_metrics):

    plt.rcParams["font.size"] = 12
    fig, axes = plt.subplots(3, 3, sharex=True, sharey=True, figsize=[10, 7])

    for i_row, (kind, vmax) in enumerate(
        zip(
            ["Normal", "Uniform", "Lognormal"],
            [0.1, 0.1, 0.1],
        )
    ):

        for j_col, metric_name in enumerate(["gamma_std", "gamma_sane", target_metric]):
            ax = axes[i_row, j_col]

            lst = []
            for seed in [221958, 771155, 231932, 465838, 359178]:
                ref_experiment_name = f"{kind}-Ens200-OutFalse-Ntime12000_Seed{seed:07}"
                lst.append(
                    np.array(
                        read_pickle(f"{RESULT_DIR}/{ref_experiment_name}.pkl")[
                            metric_name
                        ]
                    )
                )
            assert len(lst) == 5
            ref_data = np.mean(np.stack(lst), axis=0)
            assert ref_data.ndim == 1 and ref_data.shape == (118,)

            xs = np.array([10, 20, 40, 60, 80, 100, 150, 200])
            ys = np.array([120, 240, 600, 1_200, 2_400, 6_000, 12_000])
            diffs = np.zeros((len(xs), len(ys)))

            for (i, x), (j, y) in product(enumerate(xs), enumerate(ys)):

                lst = []
                for seed in [221958, 771155, 231932, 465838, 359178]:
                    experiment_name = (
                        f"{kind}-Ens{x:03}-Out{is_outlier}-Ntime{y:05}_Seed{seed:07}"
                    )
                    lst.append(
                        np.array(
                            read_pickle(f"{RESULT_DIR}/{experiment_name}.pkl")[
                                metric_name
                            ]
                        )
                    )
                assert len(lst) == 5
                data = np.mean(np.stack(lst), axis=0)
                assert data.ndim == 1 and data.shape == ref_data.shape

                corrs = list(read_pickle(f"{RESULT_DIR}/{experiment_name}.pkl")["corr"])
                assert corrs[0] == 0.0
                dx = np.diff([-corrs[1]] + corrs)
                assert np.sum(dx) == 1.0

                tmp = np.abs(data - ref_data)
                assert tmp.shape == dx.shape == data.shape == ref_data.shape == (118,)
                diffs[i, j] = np.sum(dx * tmp)

            print(metric_name, is_outlier, np.nanquantile(diffs, 0.95))

            X, Y = np.meshgrid(xs, ys / 12.0, indexing="ij")
            cnts = ax.pcolormesh(
                np.log10(X),
                np.log10(Y),
                diffs,
                vmin=0.0,
                vmax=vmax,
                shading="nearest",
                edgecolors="gray",
                cmap=cmap,
                linewidth=0.05,
            )
            fig.colorbar(cnts, ax=ax, extend="max", ticks=np.linspace(0, vmax, 6))

            ax.set_yticks(
                np.log10(ys / 12.0),
                labels=pd.Series(ys / 12.0).apply(lambda y: f"{int(y):,}"),
            )
            ax.set_xticks(
                np.log10([10, 20, 40, 60, 100, 200]), labels=[10, 20, 40, 60, 100, 200]
            )
            ttl = dict_metric_names[metric_name]
            knd = kind if kind != "Normal" else "Gaussian"
            ax.set_title(f"{ttl} ({knd} Dist.)")
            if j_col == 0:
                ax.set_ylabel(r"$T$, Data Length [yr]")

            if i_row == 2:
                ax.set_xlabel(r"$N_{\rm ens}$, Ensemble Size")

    plt.tight_layout()
    plt.show()
    if target_metric == "gamma_w3" and is_outlier:
        fig.savefig(f"{FIG_DIR}/fig03.jpg", bbox_inches="tight")
    elif target_metric == "gamma_w3" and not is_outlier:
        fig.savefig(f"{FIG_DIR}/fig08.jpg", bbox_inches="tight")

# Corr vs Metric


In [ ]:
dict_metric_names = {
    "gamma_std": r"$\gamma_\mathrm{var}$",
    "gamma_sane": r"$\gamma_\mathrm{info}$",
    "gamma_sane_original": r"$\gamma_\mathrm{info,org}$",
    "gamma_sane_original_with_fixed_params": r"$\gamma_\mathrm{info,org,fixed}$",
    "gamma_w1": r"$\gamma_{W_1,{\rm pot}}$",
    "gamma_w2": r"$\gamma_{W_2,{\rm pot}}$",
    "gamma_w3": r"$\gamma_{W_1}$",
    "gamma_w4": r"$\gamma_{W_2}$",
    "gamma_w5": r"$\widetilde{\gamma}_{W_1}$",
}

for kind in ["Normal", "Uniform", "Lognormal"]:

    if "Trend" in kind and target_metric.endswith("_with_fixed_params"):
        continue

    plt.rcParams["font.size"] = 13
    fig, axes = plt.subplots(3, 3, sharex=True, sharey=True, figsize=[12, 7])
    legend_x = 1.9

    for ax in np.ravel(axes):
        ax.set_xlim(-0.02, 1)
        ax.set_ylim(0, 1.05)
        ax.set_xticks([0, 0.5, 1])
        ax.set_yticks([0, 0.5, 1])

    n_times = 12_000
    n_ens = 40
    for ax, metric_name in zip(axes[0], ["gamma_std", "gamma_sane", target_metric]):
        for is_outlier, ls in zip([False, True], ["-", "--"]):
            if "Trend" in kind and is_outlier:
                continue

            lst = []
            for seed in [221958, 771155, 231932, 465838, 359178]:
                experiment_name = f"{kind}-Ens{n_ens:03}-Out{is_outlier}-Ntime{n_times:05}_Seed{seed:07}"
                lst.append(
                    read_pickle(f"{RESULT_DIR}/{experiment_name}.pkl")[metric_name]
                )
            assert len(lst) == 5
            ys = np.mean(np.stack(lst), axis=0)
            xs = list(read_pickle(f"{RESULT_DIR}/{experiment_name}.pkl")["corr"])
            assert ys.shape == (118,) and len(xs) == 118

            ax.plot(
                xs,
                ys,
                label=f"{'With' if is_outlier else 'Without'} Outlier",
                ls=ls,
                color="#ABABAB",
                lw=2.5 if (not is_outlier) else 1.5,
            )

            ttl = dict_metric_names[metric_name]
            ttl = f"{ttl},\n" + r"$N_{\rm ens} =$" + f"{n_ens:,}"
            ttl = f"{ttl}, " + r"$T =$" + f"{int(n_times / 12.0):,} yr"

            ax.set_title(ttl)
            ax.set_ylabel(dict_metric_names[metric_name])
    axes[0, -1].legend(
        loc="lower right", bbox_to_anchor=(legend_x, 0), edgecolor="k", ncol=1
    )

    n_times = 12_000
    is_outlier = False
    for ax, metric_name in zip(axes[1], ["gamma_std", "gamma_sane", target_metric]):
        for n_ens, color in zip(
            reversed([10, 20, 40, 200]),
            reversed(["#006BA4", "#FF800E", "#ABABAB", "#595959"]),
        ):
            lst = []
            for seed in [221958, 771155, 231932, 465838, 359178]:
                experiment_name = f"{kind}-Ens{n_ens:03}-Out{is_outlier}-Ntime{n_times:05}_Seed{seed:07}"
                lst.append(
                    read_pickle(f"{RESULT_DIR}/{experiment_name}.pkl")[metric_name]
                )
            assert len(lst) == 5
            ys = np.mean(np.stack(lst), axis=0)
            xs = list(read_pickle(f"{RESULT_DIR}/{experiment_name}.pkl")["corr"])
            assert ys.shape == (118,) and len(xs) == 118

            ax.plot(
                xs,
                ys,
                label=r"$N_{\rm ens}=$" + f"{n_ens}",
                color=color,
                ls="-",
                lw=2.5 if n_ens == 40 else 1.5,
            )

            ttl = dict_metric_names[metric_name]
            ax.set_title(f"{ttl}, " + r"$T =$" + f"{int(n_times / 12.0):,} yr")
            ax.set_ylabel(dict_metric_names[metric_name])
    axes[1, -1].legend(
        loc="lower right", bbox_to_anchor=(legend_x, 0), edgecolor="k", ncol=1
    )

    n_ens = 40
    is_outlier = False
    for ax, metric_name in zip(axes[2], ["gamma_std", "gamma_sane", target_metric]):
        for n_times, color in zip(
            reversed([120, 600, 1_200, 12_000]),
            reversed(["#5F9ED1", "#FFBC79", "#C85200", "#ABABAB"]),
        ):
            lst = []
            for seed in [221958, 771155, 231932, 465838, 359178]:
                experiment_name = f"{kind}-Ens{n_ens:03}-Out{is_outlier}-Ntime{n_times:05}_Seed{seed:07}"
                lst.append(
                    read_pickle(f"{RESULT_DIR}/{experiment_name}.pkl")[metric_name]
                )
            assert len(lst) == 5
            ys = np.mean(np.stack(lst), axis=0)
            xs = list(read_pickle(f"{RESULT_DIR}/{experiment_name}.pkl")["corr"])
            assert ys.shape == (118,) and len(xs) == 118

            ax.plot(
                xs,
                ys,
                label=r"$T=$" + f"{int(n_times / 12.0):,} yr",
                color=color,
                ls="-",
                lw=2.5 if n_times == 12_000 else 1.5,
            )

            ttl = dict_metric_names[metric_name]
            ax.set_title(f"{ttl}, " + r"$N_{\rm ens} =$" + f"{n_ens:,}")
            ax.set_xlabel(r"Corr. $\rho$")
            ax.set_ylabel(dict_metric_names[metric_name])
    axes[2, -1].legend(
        loc="lower right", bbox_to_anchor=(legend_x, 0), edgecolor="k", ncol=1
    )
    knd = kind if kind != "Normal" else "Gaussian"
    plt.suptitle(f"{knd} Dist.")
    plt.tight_layout()
    plt.show()

    if target_metric == "gamma_w3":
        fig.savefig(
            f"{FIG_DIR}/toy_model_corr_vs_metric_{kind}_target{target_metric}.jpg",
            bbox_inches="tight",
        )

    if target_metric == "gamma_w3" and kind == "Normal":
        fig.savefig(f"{FIG_DIR}/fig02.jpg", bbox_inches="tight")
    elif target_metric == "gamma_w3" and kind == "Uniform":
        fig.savefig(f"{FIG_DIR}/fig06.jpg", bbox_inches="tight")
    elif target_metric == "gamma_w3" and kind == "Lognormal":
        fig.savefig(f"{FIG_DIR}/fig07.jpg", bbox_inches="tight")